# Unconstrained tooth arrangement & Form-Based Teeth Arrangement Samples

In [11]:
import os
import glob
import base64
import time
import requests
import json
import trimesh
import urllib
import numpy as np

def _create_colors():
    # 20 high contrast colors
    colors = [[230, 25, 75,255],[60, 180, 75,255],[255, 225, 25,255],\
            [0, 130, 200,255],[245, 130, 48,255],[145, 30, 180,255],[70, 240, 240,255],\
            [240, 50, 230,255],[210, 245, 60,255],[250, 190, 190,255],[0, 128, 128,255],\
            [230, 190, 255,255],[170, 110, 40,255],[255, 250, 200,255],[128, 0, 0,255],\
            [170, 255, 195,255],[128, 128, 0, 255]]
    #np.random.shuffle(colors)
    # gum color
    colors = [[255,255,255,255]] + colors
    return colors

def colored_mesh(mesh, label):
    COLORS = _create_colors()
    mcopy = mesh.copy()
    for i, l in enumerate(np.unique(label)):
        mcopy.visual.face_colors[np.where(label == l)[0]] = COLORS[i % 18]
    return mcopy

## Define call rules
Please modify the following code blocks based on the information you obtained from us.

In [12]:
# Chohotech service request URL, sent with the API documentation.
base_url = "<service request URL>"

# Chohotech file service URL, sent with the API documentation.
file_server_url = "<service file server URL>"

# The authentication header must be passed in. Please keep the TOKEN confidential!!! If it is leaked, please contact us immediately to reset it. All tasks using this TOKEN will be charged to your account.
zh_token = "<your company's service Token, sent with the contact>" # All API calls must be authenticated with the token.

user_group = "APIClient" # User group, usually named APIClient.

# Your company's user_id, sent with the API documentation.
user_id = "<your company's user_id>"

# If you have received creds.json, it will be read directly below.
if os.path.exists('../../creds.json'):
    creds = json.load(open('../../creds.json', 'r'))
    base_url = creds['base_url']
    file_server_url = creds['file_server_url']
    zh_token = creds['zh_token']
    user_id = creds['user_id']
    print("loaded creds from creds.json")

loaded creds from creds.json


In [13]:
def upload_file(file_name):
    ext = file_name.split('.')[-1]
    data = open('../../data/' + file_name, 'rb').read()
    resp = requests.get(file_server_url + f"/scratch/{user_group}/{user_id}/upload_url?" +
                        f"postfix={ext}", # Must specify postfix, i.e., file extension
                        headers={"X-ZH-TOKEN": zh_token}) # Get signed upload URL
    resp.raise_for_status()

    upload_url = resp.text[1:-1] # Returns a single string JSON "string", can also use json.loads(resp.text)

    resp = requests.put(upload_url, data) # No auth header is needed for uploading to the cloud storage service

    resp.raise_for_status()
    path = "/".join(urllib.parse.urlparse(upload_url).path.lstrip("/").split("/")[3:])
    urn = f"urn:zhfile:o:s:{user_group}:{user_id}:{path}"
    return urn

def run_job_and_get_results(json_call, timeout_sec):
    headers = {
      "Content-Type": "application/json",
      "X-ZH-TOKEN": zh_token
    }

    url = base_url + '/run'

    response = requests.request("POST", url, headers=headers, data=json.dumps(json_call))
    response.raise_for_status()
    create_result = response.json()
    run_id = create_result['run_id']
    print("workflow id is", run_id)
    url = base_url + f"/run/{run_id}"

    start_time = time.time()
    while time.time()-start_time < timeout_sec:
        time.sleep(0.3)
        response = requests.request("GET", url, headers=headers)
        result = response.json()
        if result['completed'] or result['failed']:
            break

    if not result['completed']:
        if result['failed']:
            raise ValueError("API failed due to " + str(result['reason_public']))
        raise TimeoutError("API timeout")

    print("API finished in {}s".format(time.time()-start_time))
    url = base_url + f"/data/{run_id}"
    response = requests.request("GET", url, headers=headers)
    return response.json()

def retrieve_data(urn):
    return requests.get(file_server_url + f"/file/download?" + urllib.parse.urlencode({
                        "urn": urn}),
                        headers={"X-ZH-TOKEN": zh_token}).content

def retrieve_mesh(mesh_file_json):
    resp = requests.get(file_server_url + f"/file/download?" + urllib.parse.urlencode({
                        "urn": mesh_file_json['data']}),
                        headers={"X-ZH-TOKEN": zh_token})
    return trimesh.load(trimesh.util.wrap_as_stream(resp.content), file_type=mesh_file_json['type'])

## Oral-Arrangement

Unconstrained tooth arrangement automatically arranges teeth to aesthetically pleasing positions, primarily used for marketing presentations and doctor-patient communication. "Unconstrained" means that the algorithm does not consider medical constraints provided by other data such as CBCT.

Please refer to : https://www.chohotech.com/docs/cloud-en/#/workflow/oral-arrangement-1

In [14]:
json_call = {
  "spec_group": "mesh-processing", # The invoked workflow group is sent along with the API documentation.
  "spec_name": "oral-arrangement", # The invoked workflow group is sent along with the API documentation. 
  "spec_version": "2.0-snapshot", # The invoked workflow group is sent along with the API documentation.
  "user_group": user_group,
  "user_id": user_id,
  "input_data": {
        "upper_mesh": {"type": "drc", "data": upload_file('upper_jaw_scan.drc')},
        "lower_mesh": {"type": "ply", "data": upload_file('lower_jaw_scan.ply')},
        "ipr": {"U": False, "L": False},  # Optional
        "remove_teeth_set": [],  # Optional
        "gap": []  # Optional
      },
      "output_config": {
        "teeth_comp": {"type": "ply"},
        "align_matrix": {},  
        "transformation_dict": {}  
      }
    }
result = run_job_and_get_results(json_call, 300)

workflow id is wf_1767164136-c533cb43-b826-4fb1-8d7c-794c39b45e92
API finished in 73.30726170539856s


In [15]:
print(f"Keys contained in the output: {list(result.keys())}")

Keys contained in the output: ['align_matrix', 'teeth_comp', 'transformation_dict', 'u_align_matrix']


In [16]:
# Transform each tooth in the output's teeth_comp using transformation_dict to obtain the aligned dentition result

arranged_teeth = {}
for tooth_id, mesh_info in result.get('teeth_comp', {}).items():
    mesh = retrieve_mesh(mesh_info)
    if tooth_id in result.get('transformation_dict', {}):
        matrix = np.array(result['transformation_dict'][tooth_id])
        if matrix.shape == (4, 4):
            mesh_copy = mesh.copy()
            mesh_copy.apply_transform(matrix)
            arranged_teeth[tooth_id] = mesh_copy
    else:
        arranged_teeth[tooth_id] = mesh

# show arranged tooth 
arrange_teeth = trimesh.Scene(list(arranged_teeth.values()))
arrange_teeth.show()

## oral-arrangement-with-form

Form-Based Teeth Arrangement automatically arranges teeth to positions that conform to the doctor's form requirements.

Please refer to : https://www.chohotech.com/docs/cloud-zh/#/workflow/oral-arrangement-with-form-1

You can either upload a JSON formatted form or manually fill it out as follows:

In [17]:
def example_doctor_form():  """the sample of oral-arrangement-with-form"""
    # form configuration
doctor_form = {
        "type": "U+L",
        "locked_teeth_set": [],
        "middle_line_position": {
            "U": ["U", "keep", 0],
            "L": ["L", "keep", 0]
        },
        "front_y_axis_position": None,
        "back_y_axis_position": None,
        "z_axis_position": {
            "front_teeth": ["keep", 0.0],
            "back_teeth": ["keep", 0.0]
        },
        "x_axis_position": None,
        "collision_removal": {
            "U": {"1": [], "0": []},
            "L": {"1": [], "0": []}
        },
        "init_gap": {},
        "gap": {},
        "ipr": {},
        "remove_teeth_set": [],
        "y_axis_relative_position": {
            "left": {"canine": "I", "molar": "I"},
            "right": {"canine": "I", "molar": "I"}
        },
        "front_y_axis_relative_position": ["standard", {"U": "any", "L": "any"}],
        "z_axis_relative_position": ["standard", {"front": "any", "back": "any"}],
        "back_teeth_x_axis_relative_position": False,
        "middle_line_opt": {"U": True, "L": True}
    }

In [18]:
json_call = {
  "spec_group": "mesh-processing", # The invoked workflow group is sent along with the API documentation.
  "spec_name": "oral-arrangement-with-form", # The invoked workflow group is sent along with the API documentation. 
  "spec_version": "1.0-snapshot", # The invoked workflow group is sent along with the API documentation.
  "user_group": user_group,
  "user_id": user_id,
  "input_data": {
        "upper_mesh": {"type": "drc", "data": upload_file('upper_jaw_scan.drc')},
        "lower_mesh": {"type": "ply", "data": upload_file('lower_jaw_scan.ply')},
        "form": json.dumps(doctor_form)  # The form must be a JSON string
      },
  "output_config": {
        "teeth_comp": {"type": "ply"},        # After teeth completion
        "arranged_comp": {"type": "ply"},     # Teeth after arrangement
        "upper_mesh": {"type": "ply"},        # Preprocessed upper jaw
        "lower_mesh": {"type": "ply"},        # Preprocessed lower jaw
        "align_matrix": {},                   # A registration transformation matrix
        "transformation_dict": {}             # Individual tooth transformation matrix
  }
}
result = run_job_and_get_results(json_call, 300)

workflow id is wf_1767164220-c5f222e1-4930-4788-afdb-079897260502
API finished in 101.11654496192932s


In [19]:
print(f"Keys in the output: {list(result.keys())}")

Keys in the output: ['align_matrix', 'arranged_comp', 'lower_mesh', 'lower_seg_label', 'teeth_comp', 'transformation_dict', 'u_align_matrix', 'upper_mesh', 'upper_seg_label']


In [20]:
# show arranged tooth
sum([retrieve_mesh(result['arranged_comp'][k]) for k in result['arranged_comp'].keys()], None).show()